In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler
import joblib

BASE = Path(".") 

# Load Data
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

# Build Base Splits (cause No inplace drops on raw data)
df_train = pd.concat([df24, df25], ignore_index=True)
df_test  = df26.copy()

LEAK = ["Annual_Rounds", "Months_In_Season", "Year", "Season",
        "Division", "Field_No", "Target_Lag1", "Target_Lag2"]
# Drop constants and missing test data explicitly
DROP_EXTRA = ["Solar_Current", "Rainfall_Current", "WetDays_Current"]

df_train.drop(columns=[c for c in LEAK + DROP_EXTRA if c in df_train.columns], inplace=True)
df_test.drop(columns=[c for c in LEAK + DROP_EXTRA if c in df_test.columns], inplace=True)

# Filter Pruning Anomalies
df_train = df_train[df_train["Near_Pruning_Flag"] == 0].reset_index(drop=True)
df_test  = df_test[df_test["Near_Pruning_Flag"] == 0].reset_index(drop=True)
df_train.drop(columns=["Near_Pruning_Flag"], inplace=True)
df_test.drop(columns=["Near_Pruning_Flag"],  inplace=True)

# Feature Engineering
def engineer(df):
    df = df.copy()
    df["Soil_Index"] = df["Soil_Carbon"] / (df["Soil_pH"].replace(0, np.nan) + 1e-9)
    df["Yield_Eff"] = df["Yield_Prev_Year"] / (df["Extent_Hect"].replace(0, np.nan) + 1e-9)
    df["Prune_Age_Ratio"] = df["Prune_Cycle_Stage"] / (df["Age_Months"] / 12 + 1e-9)
    # Bug 2: Removed duplicated Rain_Intensity1 logic
    if "Rainfall_Lag1" in df.columns and "Rainfall_Lag3" in df.columns:
        df["Rain_Trend"] = df["Rainfall_Lag1"] - df["Rainfall_Lag3"]
    if "Growth_Response" in df.columns and "Field_Productivity" in df.columns:
        df["Growth_per_Prod"] = df["Growth_Response"] / (df["Field_Productivity"] + 1e-9)
    return df

df_train = engineer(df_train)
df_test  = engineer(df_test)

# Separate Targets
TARGET = "Target_Days"
X_tr_raw = df_train.drop(columns=[TARGET])
y_tr     = df_train[TARGET].astype(float).values
X_te_raw = df_test.drop(columns=[TARGET])
y_te     = df_test[TARGET].astype(float).values

#  Imputation & Normalisation
num_cols = X_tr_raw.select_dtypes(include=[np.number]).columns.tolist()

#  Outlier Clipping [1st - 99th percentile]
lo = X_tr_raw[num_cols].quantile(0.01)
hi = X_tr_raw[num_cols].quantile(0.99)
X_tr_clip = X_tr_raw.copy(); X_tr_clip[num_cols] = X_tr_raw[num_cols].clip(lo, hi, axis=1)
X_te_clip = X_te_raw.copy(); X_te_clip[num_cols] = X_te_raw[num_cols].clip(lo, hi, axis=1)

# Target Normalisation (0 to 1)
t_scaler = MinMaxScaler()
y_tr_n   = t_scaler.fit_transform(y_tr.reshape(-1, 1)).ravel()

# KNNImputer and MinMaxScaler are now saved un-fitted, to be used inside the CV loop
base_pipe = Pipeline([("imp", KNNImputer(n_neighbors=5)), ("sc", MinMaxScaler())])

out_file = BASE / "02_preprocessed_data.pkl"
joblib.dump({
    'X_tr_clip': X_tr_clip[num_cols], 'y_tr': y_tr, 'y_tr_n': y_tr_n,
    'X_te_clip': X_te_clip[num_cols], 'y_te': y_te,
    'base_pipe': base_pipe, 't_scaler': t_scaler,
    'num_cols': num_cols, 'lo': lo, 'hi': hi
}, out_file)

print(f"Final Train Shape: {X_tr_clip.shape} features, {len(y_tr)} targets")
print(f"Final Test Shape : {X_te_clip.shape} features, {len(y_te)} targets")
print(f"Saved checkpoint to: {out_file.name}")



Final Train Shape: (141, 36) features, 141 targets
Final Test Shape : (62, 36) features, 62 targets
Saved checkpoint to: 02_preprocessed_data.pkl
